# Gold - dim_equipment

This notebook loads `amperedrive_giga.gold.dim_equipment` from `amperedrive_giga.silver.dim_equipments`, one row per machine or robot asset (`equipment_id`). Load pattern: Full Overwrite / SCD Type 1. A full-sync MERGE updates changed rows, inserts new ones, and deletes assets that are no longer in silver. The surrogate key is `xxhash64(CAST(equipment_id AS STRING))`.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS amperedrive_giga.gold.dim_equipment (
  equipment_sk      BIGINT    NOT NULL COMMENT 'PK - xxhash64(equipment_id)',
  equipment_id      INT       NOT NULL COMMENT 'NK',
  plant_id          INT       NOT NULL COMMENT 'FK - plant natural key',
  equipment_name    STRING    NOT NULL,
  equipment_type    STRING    NOT NULL,
  model_number      STRING,
  source_created_at TIMESTAMP NOT NULL,
  source_updated_at TIMESTAMP NOT NULL,
  gold_load_ts      TIMESTAMP NOT NULL,
  CONSTRAINT pk_dim_equipment PRIMARY KEY (equipment_sk)
)
COMMENT 'Gold equipment dimension (SCD Type 1) sourced from silver.dim_equipments'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW vw_src_dim_equipment AS
SELECT
  xxhash64(CAST(equipment_id AS STRING)) AS equipment_sk,
  equipment_id,
  plant_id,
  TRIM(equipment_name)                   AS equipment_name,
  TRIM(equipment_type)                   AS equipment_type,
  TRIM(model_number)                     AS model_number,
  created_at                             AS source_created_at,
  updated_at                             AS source_updated_at
FROM amperedrive_giga.silver.dim_equipments
WHERE equipment_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY equipment_id ORDER BY updated_at DESC NULLS LAST) = 1

In [0]:
%sql
MERGE INTO amperedrive_giga.gold.dim_equipment AS t
USING vw_src_dim_equipment AS s
ON t.equipment_id = s.equipment_id
WHEN MATCHED AND (
     NOT (t.plant_id          <=> s.plant_id)
  OR NOT (t.equipment_name    <=> s.equipment_name)
  OR NOT (t.equipment_type    <=> s.equipment_type)
  OR NOT (t.model_number      <=> s.model_number)
  OR NOT (t.source_created_at <=> s.source_created_at)
  OR NOT (t.source_updated_at <=> s.source_updated_at)
) THEN UPDATE SET
  t.plant_id          = s.plant_id,
  t.equipment_name    = s.equipment_name,
  t.equipment_type    = s.equipment_type,
  t.model_number      = s.model_number,
  t.source_created_at = s.source_created_at,
  t.source_updated_at = s.source_updated_at,
  t.gold_load_ts      = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
  equipment_sk, equipment_id, plant_id, equipment_name, equipment_type, model_number,
  source_created_at, source_updated_at, gold_load_ts
) VALUES (
  s.equipment_sk, s.equipment_id, s.plant_id, s.equipment_name, s.equipment_type, s.model_number,
  s.source_created_at, s.source_updated_at, CURRENT_TIMESTAMP()
)
-- Full-overwrite semantics: remove equipment no longer present in silver
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM amperedrive_giga.gold.dim_equipment)                        AS gold_rows,
  (SELECT COUNT(DISTINCT equipment_id) FROM amperedrive_giga.silver.dim_equipments) AS silver_distinct_equipment,
  assert_true(
    (SELECT COUNT(*) - COUNT(DISTINCT equipment_sk) FROM amperedrive_giga.gold.dim_equipment) = 0,
    'dim_equipment: duplicate equipment_sk') AS dq_unique_sk,
  assert_true(
    (SELECT COUNT(*) FROM amperedrive_giga.gold.dim_equipment) =
    (SELECT COUNT(DISTINCT equipment_id) FROM amperedrive_giga.silver.dim_equipments WHERE equipment_id IS NOT NULL),
    'dim_equipment: row count does not match silver') AS dq_row_count